# Part 3(a) results analysis

Parses `mcperf_*.txt` and `pods_*.json` (batch timing from pod `containerStatuses[0].state.terminated.{startedAt,finishedAt}`, not Job YAML). **x = 0** = min **startedAt** over `parsec-*` pods. **Top:** mcperf bars + SLO. **Bottom:** `job-name`, `cca-project-nodetype`, `taskset -c` from pod spec.

**SLO table** (after all plots): mcperf overlapping `[min container startedAt, max finishedAt]`; `run`, `makespan_s`, `datapoints`, `violations`, `slo_ratio`; violation if `p95_ms > 1.0`.

In [2]:
# Part 3(a) — automated analysis of mcperf + batch job placement

from __future__ import annotations

import re
import json
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Project root: directory that contains part3_a_results/
_cwd = Path.cwd()
if (_cwd / "part3_a_results").is_dir():
    ROOT = _cwd
elif (_cwd.parent / "part3_a_results").is_dir():
    ROOT = _cwd.parent
else:
    ROOT = _cwd

RUNS = [
    (1, ROOT / "part3_a_results/run_1/mcperf_1.txt", ROOT / "part3_a_results/run_1/pods_1.json",
     ROOT / "part3_a_results/run_1/part3a_run1_plot.png"),
    (2, ROOT / "part3_a_results/run_2/mcperf_2.txt", ROOT / "part3_a_results/run_2/pods_2.json",
     ROOT / "part3_a_results/run_2/part3a_run2_plot.png"),
    (3, ROOT / "part3_a_results/run_3/mcperf_3.txt", ROOT / "part3_a_results/run_3/pods_3.json",
     ROOT / "part3_a_results/run_3/part3a_run3_plot.png"),
]

JOB_COLORS = {
    "barnes": "#AACCCA",
    "blackscholes": "#CCA000",
    "canneal": "#CCCCAA",
    "freqmine": "#0CCA00",
    "radix": "#00CCA0",
    "streamcluster": "#CCACCA",
    "vips": "#CC0A00",
}

SLO_MS = 1.0


def parse_mcperf(path: Path) -> pd.DataFrame:
    lines = path.read_text(encoding="utf-8", errors="replace").splitlines()
    if not lines:
        raise ValueError(f"{path}: empty file")
    header_line = lines[0].strip()
    if not header_line.startswith("#"):
        raise ValueError(f"{path}: expected # header line (mcperf)")
    colnames = header_line.lstrip("#").split()
    df = pd.read_csv(path, sep=r"\s+", skiprows=1, names=colnames, engine="python")
    df.columns = [str(c).strip() for c in df.columns]
    for col in ("p95", "ts_start", "ts_end"):
        if col not in df.columns:
            raise ValueError(f"{path}: missing column {col!r}; have {list(df.columns)}")
    df = df.astype({"p95": float, "ts_start": np.int64, "ts_end": np.int64})
    df["p95_ms"] = df["p95"] / 1000.0
    df["ts_start_s"] = df["ts_start"].astype(np.float64) / 1000.0
    df["ts_end_s"] = df["ts_end"].astype(np.float64) / 1000.0
    return df


def _parse_k8s_time(s: str) -> datetime:
    s = s.strip()
    fmts = (
        "%Y-%m-%dT%H:%M:%SZ",
        "%Y-%m-%dT%H:%M:%S.%fZ",
        "%a, %d %b %Y %H:%M:%S %z",
        "%Y-%m-%dT%H:%M:%S%z",
    )
    last_err = None
    for fmt in fmts:
        try:
            d = datetime.strptime(s, fmt)
            if fmt.endswith("Z") and d.tzinfo is None:
                d = d.replace(tzinfo=timezone.utc)
            return d
        except ValueError as e:
            last_err = e
    # Pandas fallback for slight format differences
    try:
        ts = pd.to_datetime(s, utc=True)
        d = ts.to_pydatetime()
        if d.tzinfo is None:
            d = d.replace(tzinfo=timezone.utc)
        return d
    except Exception as e:
        last_err = e
    raise ValueError(f"Could not parse time: {s!r}: {last_err}")


def _expand_taskset_cores(spec: str) -> list[int]:
    cores: list[int] = []
    spec = spec.strip()
    for part in re.split(r"\s*,\s*", spec):
        part = part.strip()
        if not part:
            continue
        m = re.match(r"^(\d+)-(\d+)$", part)
        if m:
            a, b = int(m.group(1)), int(m.group(2))
            lo, hi = (a, b) if a <= b else (b, a)
            cores.extend(range(lo, hi + 1))
            continue
        if re.match(r"^\d+$", part):
            cores.append(int(part))
            continue
    return sorted(set(cores))


def _machine_from_pod(pod: dict) -> str:
    spec = pod.get("spec") or {}
    ns = spec.get("nodeSelector") or {}
    v = ns.get("cca-project-nodetype")
    if v:
        return str(v)
    nn = str(spec.get("nodeName") or "")
    if "node-a-8core" in nn:
        return "node-a-8core"
    if "node-b-4core" in nn:
        return "node-b-4core"
    return nn or "unknown"


def _taskset_spec_from_pod(pod: dict) -> str | None:
    for c in pod.get("spec", {}).get("containers") or []:
        for arg in c.get("args") or []:
            if not isinstance(arg, str):
                continue
            if "taskset" not in arg or "-c" not in arg:
                continue
            m = re.search(r"taskset\s+-c\s+([^\s]+)", arg)
            if m:
                return m.group(1)
    return None


def parse_pods_json(path: str | Path) -> pd.DataFrame:
    """Batch job timing from pod container *terminated* timestamps (actual runtime)."""
    raw = json.loads(Path(path).read_text(encoding="utf-8", errors="replace"))
    rows: list[dict] = []
    for pod in raw.get("items") or []:
        meta = pod.get("metadata") or {}
        pname = str(meta.get("name") or "")
        if not pname.startswith("parsec-"):
            continue
        labels = meta.get("labels") or {}
        job_name = labels.get("job-name")
        if not job_name or not str(job_name).startswith("parsec-"):
            continue
        job_name = str(job_name)
        st = pod.get("status") or {}
        cstats = st.get("containerStatuses") or []
        if not cstats:
            continue
        cs0 = cstats[0]
        term = (cs0.get("state") or {}).get("terminated")
        if not term:
            continue
        sa = term.get("startedAt")
        fa = term.get("finishedAt")
        if not sa or not fa:
            continue
        start_dt = _parse_k8s_time(str(sa))
        end_dt = _parse_k8s_time(str(fa))
        machine = _machine_from_pod(pod)
        ts_sp = _taskset_spec_from_pod(pod)
        if not ts_sp:
            continue
        cores = _expand_taskset_cores(ts_sp)
        if not cores:
            continue
        short = job_name[len("parsec-") :] if job_name.lower().startswith("parsec-") else job_name
        rows.append(
            {
                "job_name": job_name,
                "short_name": short,
                "start_dt": start_dt,
                "end_dt": end_dt,
                "machine": machine,
                "cores": cores,
            }
        )
    return pd.DataFrame(rows)


def job_short_color(job_name: str) -> tuple[str, str]:
    s = job_name[len("parsec-") :] if job_name.lower().startswith("parsec-") else job_name
    c = JOB_COLORS.get(s.lower(), "#888888")
    return c, s


def relative_seconds(t0: datetime, dt: datetime) -> float:
    return (dt - t0).total_seconds()


def mcperf_overlaps_window(ts_start_s: float, ts_end_s: float, win_lo: float, win_hi: float) -> bool:
    """True iff mcperf interval overlaps [first_batch_job_start, last_batch_job_end] (container runtimes)."""
    return ts_end_s >= win_lo and ts_start_s <= win_hi


def plot_run(
    run_id: int,
    mcperf_df: pd.DataFrame,
    jobs_df: pd.DataFrame,
    t0: datetime,
    first_batch_epoch_s: float,
    win_hi_s: float,
    out_path: Path,
):
    fig, (ax1, ax2) = plt.subplots(
        2,
        1,
        figsize=(14, 10),
        gridspec_kw={"height_ratios": [2.4, 2.0]},
        sharex=True,
        constrained_layout=True,
    )

    p95_top = float(mcperf_df["p95_ms"].max())
    y_lim_top = max(1.2, p95_top * 1.2)
    ax1.set_ylim(0, y_lim_top)

    jobs_ann = jobs_df.sort_values("start_dt").reset_index(drop=True)

    # Colored vertical dashed lines at job start/end (no text; names on lower panel)
    for i in range(len(jobs_ann)):
        job = jobs_ann.iloc[i]
        color, _ = job_short_color(job["job_name"])
        t_s = relative_seconds(t0, job["start_dt"])
        t_e = relative_seconds(t0, job["end_dt"])
        ax1.axvline(t_s, color=color, linestyle="--", linewidth=0.9, alpha=0.95, zorder=2)
        ax1.axvline(t_e, color=color, linestyle="--", linewidth=0.9, alpha=0.95, zorder=2)

    # One mcperf row => one bar; width = actual measurement interval in seconds
    for _, r in mcperf_df.iterrows():
        ts_start_ms = float(r["ts_start"])
        ts_end_ms = float(r["ts_end"])
        x_start = (ts_start_ms / 1000.0) - first_batch_epoch_s
        width = (ts_end_ms - ts_start_ms) / 1000.0
        p95_ms = float(r["p95_ms"])
        ax1.bar(
            x_start,
            p95_ms,
            width=width,
            align="edge",
            color="#4A90C4",
            edgecolor="white",
            linewidth=0.3,
            alpha=0.9,
            zorder=5,
        )

    ax1.axhline(SLO_MS, color="red", linestyle="--", linewidth=1.5, label="SLO (1 ms)", zorder=6)
    ax1.grid(axis="y", alpha=0.3)
    ax1.set_ylabel("p95 latency (ms)")
    ax1.set_title(f"Run {run_id}: memcached p95 latency and batch job placement")
    ax1.legend(loc="upper right")

    # --- Core timeline: per (machine, core), stack overlapping jobs; shorter bars on top ---
    pairs: list[tuple[str, int]] = []
    for _, job in jobs_df.iterrows():
        mach = job["machine"]
        for c in job["cores"]:
            pairs.append((mach, c))
    pairs = sorted(set(pairs), key=lambda x: (x[0], x[1]))
    y_of = {p: float(i) for i, p in enumerate(pairs)}
    bar_h = 0.68
    y_stack_scale = 0.55

    def layout_overlapping_intervals(
        blocks: list[tuple[float, float, object]],
    ) -> list[tuple[float, float, object, int, int]]:
        blocks_sorted = sorted(blocks, key=lambda b: b[0])
        layer_ends: list[float] = []
        rows: list[tuple[float, float, object, int]] = []
        for t_rel0, t_rel1, job in blocks_sorted:
            t_rel1 = max(t_rel1, t_rel0 + 1e-6)
            chosen: int | None = None
            for L in range(len(layer_ends)):
                if t_rel0 >= layer_ends[L] - 1e-9:
                    chosen = L
                    layer_ends[L] = t_rel1
                    break
            if chosen is None:
                chosen = len(layer_ends)
                layer_ends.append(t_rel1)
            rows.append((t_rel0, t_rel1, job, chosen))
        n_layers = len(layer_ends)
        return [(a, b, j, L, n_layers) for (a, b, j, L) in rows]

    blocks_by_pair: dict[tuple[str, int], list[tuple[float, float, object]]] = defaultdict(list)
    for _, job in jobs_df.iterrows():
        mach = job["machine"]
        ta = relative_seconds(t0, job["start_dt"])
        tb = relative_seconds(t0, job["end_dt"])
        for core in job["cores"]:
            blocks_by_pair[(mach, core)].append((ta, tb, job))

    to_draw: list[dict] = []
    for (mach, core), blocks in sorted(blocks_by_pair.items()):
        laid = layout_overlapping_intervals(blocks)
        y_row = y_of[(mach, core)]
        for t_rel0, t_rel1, job, L, n_layers in laid:
            w = max(t_rel1 - t_rel0, 1e-6)
            color, _ = job_short_color(job["job_name"])
            if n_layers > 1:
                y_center = y_row + (L - 0.5 * (n_layers - 1)) * y_stack_scale
                h_use = bar_h * 0.68
            else:
                y_center = y_row
                h_use = bar_h
            to_draw.append(
                {
                    "t0": t_rel0,
                    "t1": t_rel1,
                    "w": w,
                    "y": y_center,
                    "h": h_use,
                    "color": color,
                    "job": job,
                }
            )

    # Narrow jobs drawn last (higher z) so radix/vips both stay visible when close
    to_draw.sort(key=lambda d: d["w"])
    for zi, d in enumerate(to_draw):
        ax2.barh(
            d["y"],
            d["w"],
            left=d["t0"],
            height=d["h"],
            color=d["color"],
            edgecolor="black",
            linewidth=0.5,
            alpha=0.9,
            zorder=4 + zi * 0.04,
        )

    LABEL_MIN_WIDTH_S = 12.0
    for zi, d in enumerate(to_draw):
        name = str(d["job"]["job_name"])
        t_rel0, w = d["t0"], d["w"]
        t_rel1 = d["t1"]
        y_center = d["y"]
        fs = 7 if w >= LABEL_MIN_WIDTH_S else 6.5
        if w >= LABEL_MIN_WIDTH_S:
            ax2.text(
                t_rel0 + w / 2,
                y_center,
                name,
                ha="center",
                va="center",
                fontsize=fs,
                color="black",
                zorder=50 + zi,
            )
        else:
            ax2.text(
                t_rel1 + 0.6,
                y_center,
                name,
                ha="left",
                va="center",
                fontsize=fs,
                color="black",
                zorder=50 + zi,
                clip_on=False,
            )

    ax2.set_yticks(range(len(pairs)))
    ax2.set_yticklabels([f"{m} core {c}" for m, c in pairs])
    ax2.set_xlabel("Time (s) from first batch container start (pod terminated timestamps)")
    ax2.set_ylabel("CPU core")
    ax2.invert_yaxis()

    win_span = win_hi_s - first_batch_epoch_s
    mc_end_rel = (mcperf_df["ts_end"].astype(np.float64) / 1000.0) - first_batch_epoch_s
    xmax = float(max(win_span, mc_end_rel.max(), 1.0))
    ax2.set_xlim(0, xmax * 1.02)
    ax1.set_xlim(0, xmax * 1.02)

    fig.savefig(out_path, dpi=150, bbox_inches="tight", pad_inches=0.45)
    plt.close(fig)
    print(f"Saved plot: {out_path}")


# --- Main loop ---
summary_rows = []

for run_id, mc_path, pods_path, plot_path in RUNS:
    mc = parse_mcperf(mc_path)
    jobs = parse_pods_json(pods_path)
    if jobs.empty:
        print(f"Run {run_id}: no PARSEC pods parsed from {pods_path}")
        continue

    t0 = jobs["start_dt"].min()
    win_end = jobs["end_dt"].max()
    win_lo_s = t0.timestamp()
    win_hi_s = win_end.timestamp()

    mask = mc.apply(
        lambda r: mcperf_overlaps_window(r["ts_start_s"], r["ts_end_s"], win_lo_s, win_hi_s),
        axis=1,
    )
    mc_win = mc.loc[mask].copy()
    violations = int((mc_win["p95_ms"] > 1.0).sum())
    n_in = int(len(mc_win))
    ratio = float(violations / n_in) if n_in else float("nan")

    plot_run(run_id, mc, jobs, t0, win_lo_s, win_hi_s, plot_path)

    summary_rows.append(
        {
            "run": run_id,
            "makespan_s": win_hi_s - win_lo_s,
            "datapoints": n_in,
            "violations": violations,
            "slo_ratio": ratio,
        }
    )

summary = pd.DataFrame(summary_rows)
if not summary.empty:
    print("\nSLO summary (mcperf vs container runtime window [min startedAt, max finishedAt], p95_ms > 1.0):")
    print(summary.to_string(index=False))
else:
    print("No summary rows produced.")


Run 1 SLO violation ratio: 0.0000 (0/22 samples in window)
Saved plot: /Users/banu/Desktop/cloud-comp-arch-project/part3_a_results/run_1/part3a_run1_plot.png
Run 2 SLO violation ratio: 0.0000 (0/22 samples in window)
Saved plot: /Users/banu/Desktop/cloud-comp-arch-project/part3_a_results/run_2/part3a_run2_plot.png
Run 3 SLO violation ratio: 0.0000 (0/22 samples in window)
Saved plot: /Users/banu/Desktop/cloud-comp-arch-project/part3_a_results/run_3/part3a_run3_plot.png

Summary table:
 run           first_job_start              last_job_end  makespan_s  mcperf_in_window  slo_violations  slo_violation_ratio
   1 2026-05-10 04:26:01 +0200 2026-05-10 04:30:04 +0200       243.0                22               0                  0.0
   2 2026-05-10 04:36:34 +0200 2026-05-10 04:40:38 +0200       244.0                22               0                  0.0
   3 2026-05-10 04:55:00 +0200 2026-05-10 04:59:06 +0200       246.0                22               0                  0.0
